In [22]:
import folium
from shapely.geometry import shape
from itertools import cycle
import requests
# Biblioteca propia
from query_poligonos import extractor_poligonos_localidades,creador_poligono_municipio
from utils_poligonos import crear_diccionario_subpoligonos,filtrar_poligonos_por_distancia,creador_mapa
import matplotlib.pyplot as plt
import geopandas as gpd

## Datos

In [23]:
# Extraer el polígono Soacha
archivo_mun= r'../../data/external/MGN2023_MPIO_POLITICO/MGN_ADM_MPIO_GRAFICO.shp'
poligono_soacha=creador_poligono_municipio(path=archivo_mun)

# Extraer el polígono combinado de las localidades 'Ciudad Bolívar' y 'San Cristóbal'
poligono_cb_sc = extractor_poligonos_localidades(['Ciudad Bolívar', 'San Cristóbal'])

# Unir diccionarios
poligono_sh_cb_sc = poligono_cb_sc | poligono_soacha

# Dividir el polígono combinado en subpolígonos más pequeños de tamaño definido
subpoligonos_cb_sc_diccionario = crear_diccionario_subpoligonos(poligono_sh_cb_sc, tamano_m=200)

# Diccionario de estaciones de TransMiCable en la localidad de Potosí
transmicable_stations_potosi = {
    'PORTAl_SUR': (4.597663047655793, -74.16942291534492),  # Coordenadas del Portal Sur
    'TRES_REYES': (4.583517154195488, -74.17998854232754),  # Coordenadas de Tres Reyes
    'SANTA_VIVIANA': (4.577548820375602, -74.17632291534493),  # Coordenadas de Santa Viviana
    'POTOSI': (4.569668547284284, -74.1712978134913)  # Coordenadas de Potosí
}

# Diccionario de estaciones de TransMiCable en la localidad de Ciudad Bolívar
transmicable_stations_cb = {
    "PORTAL_DEL_TUNAL": (4.568874258616699, -74.13946745553427),  # Coordenadas del Portal del Tunal
    "JUAN_PABLO_II": (4.555992907670105, -74.14740979258688),  # Coordenadas de Juan Pablo II
    "MANITAS": (4.5505494740333985, -74.15058826560421),  # Coordenadas de Manitas
    "MIRADOR_DEL_PARAISO": (4.5502151330274065, -74.15891333669026)  # Coordenadas de Mirador del Paraíso
}

# Diccionario de estaciones de TransMiCable en la localidad de San Cristóbal
transmicable_stations_sc = {
    'PORTAL_20_DE_JULIO': (4.56516773895769, -74.09641625108391),  # Coordenadas del Portal 20 de Julio
    'LA_VICTORIA': (4.550883883505155, -74.09388602539592),  # Coordenadas de La Victoria
    'ALTAMIRA': (4.544586492181246, -74.08494407133428)  # Coordenadas de Altamira
}

# Filtrar subpolígonos en Ciudad Bolívar que estén cerca (2000 metros) de las estaciones de TransMiCable en Ciudad Bolívar
subpoligonos_cercanos_a_trasmicable_cb = filtrar_poligonos_por_distancia(
    localidad='CIUDAD_BOLIVAR',
    subpoligonos=subpoligonos_cb_sc_diccionario,
    estaciones=transmicable_stations_cb,
    distancia_maxima=2000
)

# Filtrar subpolígonos en San Cristóbal que estén cerca (2000 metros) de las estaciones de TransMiCable en San Cristóbal
subpoligonos_cercanos_a_trasmicable_sc = filtrar_poligonos_por_distancia(
    localidad='SAN_CRISTOBAL',
    subpoligonos=subpoligonos_cb_sc_diccionario,
    estaciones=transmicable_stations_sc,
    distancia_maxima=2000
)

# Filtrar subpolígonos en Ciudad Bolívar que estén cerca (2000 metros) de las estaciones de TransMiCable en Potosí
subpoligonos_cercanos_a_trasmicable_potosi_cb = filtrar_poligonos_por_distancia(
    localidad='CIUDAD_BOLIVAR',
    subpoligonos=subpoligonos_cb_sc_diccionario,
    estaciones=transmicable_stations_potosi,
    distancia_maxima=2000
)

# Filtrar subpolígonos en Soacha que estén cerca (2000 metros) de las estaciones de TransMiCable en Potosí
subpoligonos_cercanos_a_trasmicable_potosi_sh = filtrar_poligonos_por_distancia(
    localidad='SOACHA',
    subpoligonos=subpoligonos_cb_sc_diccionario,
    estaciones=transmicable_stations_potosi,
    distancia_maxima=2000
)

# Combinar los subpolígonos cercanos a las estaciones de Potosí en Ciudad Bolívar y Soacha
subpoligonos_cercanos_a_trasmicable_potosi = subpoligonos_cercanos_a_trasmicable_potosi_cb | subpoligonos_cercanos_a_trasmicable_potosi_sh

In [26]:
subpoligonos_cercanos_a_trasmicable_potosi.keys()

dict_keys(['subpoligono_469', 'subpoligono_470', 'subpoligono_471', 'subpoligono_472', 'subpoligono_545', 'subpoligono_546', 'subpoligono_547', 'subpoligono_548', 'subpoligono_549', 'subpoligono_550', 'subpoligono_551', 'subpoligono_552', 'subpoligono_635', 'subpoligono_636', 'subpoligono_637', 'subpoligono_638', 'subpoligono_639', 'subpoligono_640', 'subpoligono_641', 'subpoligono_642', 'subpoligono_643', 'subpoligono_644', 'subpoligono_645', 'subpoligono_646', 'subpoligono_729', 'subpoligono_730', 'subpoligono_731', 'subpoligono_732', 'subpoligono_733', 'subpoligono_734', 'subpoligono_735', 'subpoligono_736', 'subpoligono_737', 'subpoligono_738', 'subpoligono_739', 'subpoligono_740', 'subpoligono_741', 'subpoligono_742', 'subpoligono_743', 'subpoligono_744', 'subpoligono_745', 'subpoligono_746', 'subpoligono_747', 'subpoligono_748', 'subpoligono_749', 'subpoligono_837', 'subpoligono_838', 'subpoligono_839', 'subpoligono_840', 'subpoligono_841', 'subpoligono_842', 'subpoligono_843', '

In [21]:
for i in subpoligonos_cb_sc_diccionario.keys():
    print(i,len(subpoligonos_cb_sc_diccionario[i]))

CIUDAD_BOLIVAR 608
SAN_CRISTOBAL 242
SOACHA 855


In [16]:
len(subpoligonos_cb_sc_diccionario)

3

## Visualizacion

In [17]:
poligono_sh_cb_sc

{'CIUDAD_BOLIVAR': {'type': 'MultiPolygon',
  'coordinates': [(((-74.18135180099995, 4.588917862849369),
     (-74.18091513299993, 4.587880147849407),
     (-74.1809525999999, 4.587714362849433),
     (-74.18095259499995, 4.587593685849398),
     (-74.18085502199995, 4.587209240849401),
     (-74.18073473899994, 4.58660622184944),
     (-74.18070022099994, 4.5861570518494865),
     (-74.1806895869999, 4.5860182048494735),
     (-74.18052432299993, 4.584616098849518),
     (-74.18044916999992, 4.583975383849507),
     (-74.18035907799992, 4.583673920849548),
     (-74.18028384299993, 4.583213994849541),
     (-74.17999838199995, 4.580994090849658),
     (-74.17992321799994, 4.580010326849677),
     (-74.17970144499992, 4.578366589849738),
     (-74.1795795509999, 4.577795253849749),
     (-74.17955527599992, 4.577694617849739),
     (-74.17952368599992, 4.577534151849731),
     (-74.1794140419999, 4.577115373849762),
     (-74.17939343799992, 4.576999014849764),
     (-74.17939341899995

In [5]:
creador_mapa(diccionario_poligonos=poligono_sh_cb_sc)

In [6]:
creador_mapa(diccionario_poligonos=poligono_sh_cb_sc,diccionario_sub_poligonos=subpoligonos_cb_sc_diccionario)

In [18]:
subpoligonos_cb_sc_diccionario.keys()

dict_keys(['CIUDAD_BOLIVAR', 'SAN_CRISTOBAL', 'SOACHA'])

In [8]:
# Diccionario de estaciones de TransMiCable en la localidad de Potosí
transmicable_stations_potosi = {
    'PORTAl_SUR': (4.597663047655793, -74.16942291534492),  # Coordenadas del Portal Sur
    'TRES_REYES': (4.583517154195488, -74.17998854232754),  # Coordenadas de Tres Reyes
    'SANTA_VIVIANA': (4.577548820375602, -74.17632291534493),  # Coordenadas de Santa Viviana
    'POTOSI': (4.569668547284284, -74.1712978134913)  # Coordenadas de Potosí
}

# Diccionario de estaciones de TransMiCable en la localidad de Ciudad Bolívar
transmicable_stations_cb = {
    "PORTAL_DEL_TUNAL": (4.568874258616699, -74.13946745553427),  # Coordenadas del Portal del Tunal
    "JUAN_PABLO_II": (4.555992907670105, -74.14740979258688),  # Coordenadas de Juan Pablo II
    "MANITAS": (4.5505494740333985, -74.15058826560421),  # Coordenadas de Manitas
    "MIRADOR_DEL_PARAISO": (4.5502151330274065, -74.15891333669026)  # Coordenadas de Mirador del Paraíso
}

# Diccionario de estaciones de TransMiCable en la localidad de San Cristóbal
transmicable_stations_sc = {
    'PORTAL_20_DE_JULIO': (4.56516773895769, -74.09641625108391),  # Coordenadas del Portal 20 de Julio
    'LA_VICTORIA': (4.550883883505155, -74.09388602539592),  # Coordenadas de La Victoria
    'ALTAMIRA': (4.544586492181246, -74.08494407133428)  # Coordenadas de Altamira
}

In [10]:
from itertools import zip_longest
import folium
from branca.element import Template, MacroElement

# Crear una función para definir el estilo con color dinámico
def estilo_personalizado(color):
    return lambda x: {'color': color, 'fillOpacity': 0.3}

# Crear un mapa centrado en Bogotá
m = folium.Map(location=[4.6 - 0.05, -74.1], zoom_start=13.49, tiles='cartodbpositron')

# Combinar las listas de estaciones y subpolígonos
lista_estaciones = [transmicable_stations_potosi, transmicable_stations_cb, transmicable_stations_sc]
lista_poligonos_filtrados = [subpoligonos_cercanos_a_trasmicable_potosi, subpoligonos_cercanos_a_trasmicable_cb, subpoligonos_cercanos_a_trasmicable_sc]
combinaciones = zip(lista_estaciones, lista_poligonos_filtrados)

# Iterar sobre las combinaciones de estaciones y polígonos
for estaciones, poligonos_filtrados in combinaciones:
    # Crear una lista de coordenadas para la conexión de puntos
    coords_conexion = []

    # Añadir las estaciones como puntos rojos al mapa
    for estacion, coords in estaciones.items():
        folium.CircleMarker(
            location=coords,
            radius=8,
            color='red',
            fill=True,
            fill_color='red',
            popup=folium.Popup(estacion, parse_html=True)
        ).add_to(m)
        # Añadir las coordenadas a la lista para las conexiones
        coords_conexion.append(coords)

    # Dibujar la línea punteada que conecta las estaciones
    if coords_conexion:
        folium.PolyLine(
            coords_conexion,
            color='black',
            weight=2,
            dash_array='5, 10'  # Estilo punteado
        ).add_to(m)

    # Añadir los polígonos al mapa con colores personalizados
    for subpoligono_nombre, subpoligono_geojson in poligonos_filtrados.items():
        color = 'blue'  # Color predeterminado
        if subpoligono_geojson['Estacion_cercana'] in ["PORTAL_DEL_TUNAL", "JUAN_PABLO_II","MANITAS","MIRADOR_DEL_PARAISO"]:
            color = 'green'
        folium.GeoJson(
            data=subpoligono_geojson['Dict_poligono'],
            style_function=estilo_personalizado(color),
            popup=folium.Popup(subpoligono_nombre)
        ).add_to(m)

# Añadir leyenda personalizada
legend_html = """
<div style="position: fixed;
            bottom: 50px; left: 50px; width: 200px; height: 90px;
            background-color: white; z-index:1000; padding: 10px; border: 2px solid black;">
    <h4>Colores de polígonos</h4>
    <i style="background: green; width: 18px; height: 18px; display: inline-block;"></i> Tratados<br>
    <i style="background: blue; width: 18px; height: 18px; display: inline-block;"></i> No tratados
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# Guardar el mapa como HTML
m.save(r"../../reports/figures/mapa_estimacion_1.html")
m

In [11]:
subpoligonos_cercanos=subpoligonos_cercanos_a_trasmicable_potosi| subpoligonos_cercanos_a_trasmicable_cb| subpoligonos_cercanos_a_trasmicable_sc

In [12]:
import os
import simplekml

# Ruta base donde se crearán las carpetas
base_path = r"D:\Tesis_JE_fotos"

# Crear las carpetas y archivos necesarios
for subpoligono, datos in subpoligonos_cercanos.items():
    localidad = datos['localida']
    estacion_cercana = datos['Estacion_cercana']
    nombre = datos['nombre']
    dict_poligono = datos['Dict_poligono']

    # Rutas de las carpetas
    ruta_localidad = os.path.join(base_path, localidad)
    ruta_estacion = os.path.join(ruta_localidad, estacion_cercana)
    ruta_subpoligono = os.path.join(ruta_estacion, nombre)

    # Crear las carpetas si no existen
    os.makedirs(ruta_subpoligono, exist_ok=True)

    # Crear el archivo KML en la carpeta correspondiente
    kml = simplekml.Kml()

    # Manejar Polygon y MultiPolygon
    if dict_poligono['type'] == 'Polygon':
        # Polígono simple
        coordinates = [
            (coord[0], coord[1]) for coord in dict_poligono['coordinates'][0]
        ]
        pol = kml.newpolygon(name=nombre)
        pol.outerboundaryis = coordinates
        pol.style.polystyle.color = simplekml.Color.changealphaint(0, simplekml.Color.green)  # Sin relleno
        pol.style.polystyle.fill = False  # Desactiva el relleno
        pol.style.linestyle.color = simplekml.Color.red  # Contorno rojo
        pol.style.linestyle.width = 2  # Grosor del contorno
    elif dict_poligono['type'] == 'MultiPolygon':
        # Multipolígono
        for i, multipolygon_coords in enumerate(dict_poligono['coordinates']):
            coordinates = [
                (coord[0], coord[1]) for coord in multipolygon_coords[0]
            ]
            pol = kml.newpolygon(name=f"{nombre}_part_{i+1}")
            pol.outerboundaryis = coordinates
            pol.style.polystyle.color = simplekml.Color.changealphaint(0, simplekml.Color.green)  # Sin relleno
            pol.style.polystyle.fill = False  # Desactiva el relleno
            pol.style.linestyle.color = simplekml.Color.red  # Contorno rojo
            pol.style.linestyle.width = 2  # Grosor del contorno

    # Guardar el archivo KML
    kml_file_path = os.path.join(ruta_subpoligono, f"poligono.kml")
    kml.save(kml_file_path)

    print(f"Archivo KML creado: {kml_file_path}")

    # Guardar el diccionario filtrado en un archivo de texto dentro de la carpeta
    dict_file_path = os.path.join(ruta_subpoligono, f"{nombre}_datos.txt")
    with open(dict_file_path, "w", encoding="utf-8") as f:
        f.write(str(datos))

    print(f"Archivo de diccionario creado: {dict_file_path}")

print("Proceso completado.")

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'D:\\'

In [ ]:
Diccionario_resultados={'Modelo':'U-Net_R', "Numero_total_edificios":161, 'Resultados_desglozados':{'alta_confianza':112,'media_confianza':33,'baja_confianza':16}}

In [19]:
diccionario_resultados = {
    "modelo": "u-net_r",
    "numero_total_edificios": 161,
    "resultados_desglosados": {
        "alta_confianza": 112,
        "media_confianza": 33,
        "baja_confianza": 16
    }
}

In [20]:
diccionario_resultados

{'modelo': 'u-net_r',
 'numero_total_edificios': 161,
 'resultados_desglosados': {'alta_confianza': 112,
  'media_confianza': 33,
  'baja_confianza': 16}}